# Notebook 05: Novel Candidate Screening

**Project:** GNN-Based Battery Voltage Predictor  

---

## Overview

Using the best trained model (M3GNet), this notebook screens novel Li-containing
crystal structures that do not yet have computed voltages in the Materials Project.

Screening pipeline:

1. Query Materials Project for Li-containing structures with:
   - e_above_hull less than 0.05 eV/atom (thermodynamically stable)
   - No existing electrode entry (not already in our training data)
   - Contains Li (working ion)

2. Run voltage inference with the fine-tuned M3GNet on all candidates

3. Filter and rank by:
   - Predicted voltage greater than 2.5 V vs Li/Li+ (practical cathode range)
   - Estimated stability (e_above_hull less than 50 meV/atom)
   - Formula does not contain rare/toxic elements (Hg, Tl, Pb, Cd) for feasibility

4. Output top 50 candidates as `results/top_candidates.csv`

5. Validate top 5 against any available DFT data or literature

**Note:** Inference voltage predictions for unstudied structures are extrapolations;
DFT validation is required before experimental synthesis decisions.

In [ ]:
import os
import sys
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import structure_to_graph, get_chemistry_family, set_seed
from src.evaluate import PALETTE

set_seed(42)

API_KEY = os.environ.get('MP_API_KEY', '')
if not API_KEY:
    raise EnvironmentError('MP_API_KEY not set. Export it before running.')

DATA_DIR    = project_root / 'data'
MODELS_DIR  = project_root / 'models'
RESULTS_DIR = project_root / 'results'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Query Materials Project for Novel Candidates

We query stable Li-containing compounds that are not in our training dataset.
The filter `e_above_hull < 0.05 eV/atom` selects entries on or very close to
the convex hull of thermodynamic stability. We also request the crystal structure
so we can featurize directly for GNN inference.

In [ ]:
# Load training battery IDs to exclude
with open(DATA_DIR / 'splits.pkl', 'rb') as f:
    splits = pickle.load(f)

known_formulas = set()
for split_name in ['train', 'val', 'test']:
    for e in splits[split_name]:
        known_formulas.add(e['formula'])

print(f'Known training formulas: {len(known_formulas)}')

In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Structure, Composition

CANDIDATE_CACHE = DATA_DIR / 'screening_candidates.pkl'

if CANDIDATE_CACHE.exists():
    print(f'Loading cached candidates from {CANDIDATE_CACHE}')
    with open(CANDIDATE_CACHE, 'rb') as f:
        candidates = pickle.load(f)
else:
    print('Querying Materials Project for novel Li-containing structures...')
    candidates = []

    with MPRester(api_key=API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            elements=['Li'],
            energy_above_hull=(0, 0.05),  # eV/atom
            fields=[
                'material_id', 'formula_pretty', 'energy_above_hull',
                'structure', 'symmetry', 'nsites', 'volume',
                'formation_energy_per_atom',
            ],
        )

    print(f'Retrieved {len(docs)} raw candidates')

    # Exclude rare/toxic elements
    EXCLUDE_ELEMENTS = {'Hg', 'Tl', 'Pb', 'Cd', 'Cs', 'Rb', 'Fr', 'Ra'}

    for doc in docs:
        formula = doc.formula_pretty
        # Skip entries already in training data
        family = get_chemistry_family(formula)
        if formula in known_formulas:
            continue
        # Check for Li content > 0
        try:
            comp = Composition(formula)
            if comp.get('Li', 0) == 0:
                continue
            # Check element exclusion list
            if any(str(el) in EXCLUDE_ELEMENTS for el in comp.elements):
                continue
        except Exception:
            continue

        struct = getattr(doc, 'structure', None)
        if struct is None:
            continue

        candidates.append({
            'material_id': str(doc.material_id),
            'formula': formula,
            'energy_above_hull': float(doc.energy_above_hull or 0),
            'formation_energy_per_atom': float(doc.formation_energy_per_atom or 0),
            'nsites': int(doc.nsites or 0),
            'chemistry_family': family,
            'structure_dict': struct.as_dict(),
        })

    print(f'After filtering: {len(candidates)} novel candidates')

    with open(CANDIDATE_CACHE, 'wb') as f:
        pickle.dump(candidates, f)
    print(f'Cached candidates to {CANDIDATE_CACHE}')

print(f'Total candidates for screening: {len(candidates)}')

In [ ]:
df_cand = pd.DataFrame([{
    'material_id': c['material_id'],
    'formula': c['formula'],
    'energy_above_hull_meV': c['energy_above_hull'] * 1000,
    'chemistry_family': c['chemistry_family'],
    'nsites': c['nsites'],
} for c in candidates])

print(f'Candidate dataset: {len(df_cand)} entries')
print(df_cand['chemistry_family'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].hist(df_cand['energy_above_hull_meV'], bins=40, color='#4CAF50', alpha=0.8, edgecolor='none')
axes[0].set_xlabel('E above hull (meV/atom)')
axes[0].set_ylabel('Count')
axes[0].set_title('Stability of Screening Candidates')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

chem_counts = df_cand['chemistry_family'].value_counts()
chem_colors = [PALETTE.get(f, '#607D8B') for f in chem_counts.index]
axes[1].bar(range(len(chem_counts)), chem_counts.values, color=chem_colors, alpha=0.85, edgecolor='none')
axes[1].set_xticks(range(len(chem_counts)))
axes[1].set_xticklabels(chem_counts.index, rotation=25, ha='right')
axes[1].set_ylabel('Count')
axes[1].set_title('Candidates by Chemistry Family')
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig05_candidate_overview.png')
plt.show()

## 2. Voltage Inference with Best Model (M3GNet)

We run the fine-tuned M3GNet on all candidates. Structures are featurized
using the same matgl Structure2Graph converter as in Notebook 03.

In [ ]:
import matgl
import dgl
from matgl.ext.pymatgen import Structure2Graph, get_element_list
from torch.utils.data import DataLoader as TorchDataLoader

class M3GNetRegressor(nn.Module):
    def __init__(self, backbone, hidden_dim=64):
        super().__init__()
        self.backbone = backbone
        backbone_dim = getattr(backbone.model, 'dim_node_embedding', 64)
        self.head = nn.Sequential(
            nn.Linear(backbone_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
        )
    def forward(self, g, lat, state):
        node_feats = self.backbone.model.get_graph_features(g, lat, state)
        pooled = dgl.mean_nodes(g, feat='h') if 'h' in g.ndata else node_feats.mean(0)
        return self.head(pooled).squeeze(-1)

pretrained_m3g = matgl.load_model('M3GNet-MP-2021.2.8-PES')
m3gnet_model = M3GNetRegressor(pretrained_m3g).to(device)
m3gnet_model.load_state_dict(torch.load(MODELS_DIR / 'm3gnet_best.pt', map_location=device))
m3gnet_model.eval()
print('M3GNet model loaded and ready for inference.')

In [ ]:
# Build element list from candidates + training structures
all_entries_extended = splits['train'] + splits['val'] + splits['test']
all_structs_ext = []

for e in all_entries_extended:
    sd = e.get('charged_structure') or e.get('discharged_structure')
    if sd:
        try:
            all_structs_ext.append(Structure.from_dict(sd))
        except Exception:
            pass

for c in candidates[:500]:  # Sample candidates for element list
    try:
        all_structs_ext.append(Structure.from_dict(c['structure_dict']))
    except Exception:
        pass

element_types = get_element_list(all_structs_ext)
print(f'Element types in combined vocabulary: {len(element_types)}')
converter = Structure2Graph(element_types=element_types, cutoff=5.0)

In [ ]:
from tqdm import tqdm

predicted_voltages = []
successful_indices = []

print(f'Running inference on {len(candidates)} candidates...')

batch_size = 32
batch_graphs, batch_lats, batch_states, batch_meta = [], [], [], []

def run_batch(graphs, lats, states):
    if not graphs:
        return []
    batched_g = dgl.batch(graphs).to(device)
    batched_lat = torch.stack(lats).to(device)
    batched_state = torch.stack(states).to(device) if states[0] is not None else None
    with torch.no_grad():
        preds = m3gnet_model(batched_g, batched_lat, batched_state)
    return preds.cpu().numpy().tolist()

for i, cand in enumerate(tqdm(candidates)):
    try:
        structure = Structure.from_dict(cand['structure_dict'])
        graph, lat, state = converter.get_graph(structure)
        batch_graphs.append(graph)
        batch_lats.append(lat)
        batch_states.append(state)
        batch_meta.append(i)
    except Exception:
        continue

    if len(batch_graphs) >= batch_size:
        preds = run_batch(batch_graphs, batch_lats, batch_states)
        for idx, pred in zip(batch_meta, preds):
            predicted_voltages.append((idx, pred))
        batch_graphs, batch_lats, batch_states, batch_meta = [], [], [], []

# Final partial batch
if batch_graphs:
    preds = run_batch(batch_graphs, batch_lats, batch_states)
    for idx, pred in zip(batch_meta, preds):
        predicted_voltages.append((idx, pred))

print(f'Inference complete: {len(predicted_voltages)} predictions')

In [ ]:
# Build results dataframe
screening_rows = []
for orig_idx, pred_v in predicted_voltages:
    cand = candidates[orig_idx]
    # Estimate gravimetric capacity: ~Li fraction * 26802 mAh/mol / MW
    try:
        comp = Composition(cand['formula'])
        mw = comp.weight  # g/mol
        n_li = comp.get_el_amt_dict().get('Li', 0)
        est_capacity = n_li * 26802 / mw if mw > 0 else 0
    except Exception:
        est_capacity = 0

    screening_rows.append({
        'material_id': cand['material_id'],
        'formula': cand['formula'],
        'chemistry_family': cand['chemistry_family'],
        'predicted_voltage_V': pred_v,
        'est_capacity_mAh_g': est_capacity,
        'est_energy_density_Wh_kg': pred_v * est_capacity,
        'energy_above_hull_meV': cand['energy_above_hull'] * 1000,
        'formation_energy_eV_atom': cand['formation_energy_per_atom'],
        'nsites': cand['nsites'],
    })

df_screening = pd.DataFrame(screening_rows)

# Filter: voltage > 2.5 V, stability < 50 meV/atom
df_filtered = df_screening[
    (df_screening['predicted_voltage_V'] > 2.5) &
    (df_screening['energy_above_hull_meV'] < 50)
].copy()
df_filtered = df_filtered.sort_values('predicted_voltage_V', ascending=False).reset_index(drop=True)

print(f'Candidates passing filters: {len(df_filtered)}')
print(df_filtered.head(10)[['formula', 'predicted_voltage_V', 'est_capacity_mAh_g', 'energy_above_hull_meV']].to_string(index=False))

In [ ]:
top50 = df_filtered.head(50).copy()
top50['rank'] = range(1, len(top50) + 1)

# Round for readability
for col in ['predicted_voltage_V', 'est_capacity_mAh_g', 'est_energy_density_Wh_kg', 'energy_above_hull_meV']:
    top50[col] = top50[col].round(3)

top50.to_csv(RESULTS_DIR / 'top_candidates.csv', index=False)
print(f'Saved top 50 candidates to {RESULTS_DIR / "top_candidates.csv"}')
print(top50[['rank', 'formula', 'chemistry_family', 'predicted_voltage_V', 'est_capacity_mAh_g', 'energy_above_hull_meV']].to_string(index=False))

## 3. Screening Results Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for fam, grp in df_screening.groupby('chemistry_family'):
    ax.scatter(
        grp['est_capacity_mAh_g'], grp['predicted_voltage_V'],
        s=10, alpha=0.4, label=fam,
        color=PALETTE.get(fam, '#607D8B'), linewidths=0
    )

# Highlight top 10
for _, row in top50.head(10).iterrows():
    ax.scatter(row['est_capacity_mAh_g'], row['predicted_voltage_V'],
               s=80, color='gold', marker='*', zorder=5,
               edgecolors='black', linewidths=0.5)
    ax.annotate(row['formula'], (row['est_capacity_mAh_g'], row['predicted_voltage_V']),
                fontsize=7, ha='left', va='bottom')

ax.axhline(2.5, color='grey', linestyle=':', lw=1.0, alpha=0.7, label='2.5 V cutoff')
ax.set_xlabel('Estimated Gravimetric Capacity (mAh/g)')
ax.set_ylabel('Predicted Voltage (V vs Li/Li+)')
ax.set_title('Novel Candidate Screening: Predicted Voltage vs Estimated Capacity')
ax.legend(title='Chemistry', fontsize=8, frameon=True, framealpha=0.8)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
fig.savefig(RESULTS_DIR / 'fig05_screening_scatter.png')
plt.show()

## 4. Validation of Top 5 Candidates

For the top 5 predicted candidates, we check whether any existing DFT data or
literature voltages are available in the Materials Project under a different entry
(e.g., as an electrode material with a slightly different composition).

This acts as a sanity check on the model's extrapolative performance.

In [ ]:
top5 = top50.head(5)
print('Top 5 predicted candidates for validation:')
print(top5[['rank', 'formula', 'predicted_voltage_V', 'energy_above_hull_meV']].to_string(index=False))

# Query MP for any existing electrode data on these formulas
validation_results = []

with MPRester(api_key=API_KEY) as mpr:
    for _, row in top5.iterrows():
        try:
            # Search for this formula as an electrode
            electrode_docs = mpr.electrodes.search(
                working_ion='Li',
                fields=['battery_id', 'average_voltage', 'framework_formula'],
            )
            matching = [d for d in electrode_docs
                        if getattr(d, 'framework_formula', '') == row['formula']]
            if matching:
                dft_v = matching[0].average_voltage
                validation_results.append({
                    'formula': row['formula'],
                    'predicted_voltage': row['predicted_voltage_V'],
                    'dft_voltage': dft_v,
                    'abs_error': abs(dft_v - row['predicted_voltage_V']),
                    'source': 'MP electrode database',
                })
            else:
                validation_results.append({
                    'formula': row['formula'],
                    'predicted_voltage': row['predicted_voltage_V'],
                    'dft_voltage': None,
                    'abs_error': None,
                    'source': 'No DFT reference found',
                })
        except Exception as ex:
            validation_results.append({
                'formula': row['formula'],
                'predicted_voltage': row['predicted_voltage_V'],
                'dft_voltage': None,
                'abs_error': None,
                'source': f'Query failed: {ex}',
            })

df_val = pd.DataFrame(validation_results)
print('\nValidation results:')
print(df_val.to_string(index=False))

df_val.to_csv(RESULTS_DIR / 'top5_validation.csv', index=False)

In [ ]:
print('Screening complete.')
print(f'  Total candidates screened: {len(df_screening)}')
print(f'  Passing voltage + stability filters: {len(df_filtered)}')
print(f'  Top 50 saved to: results/top_candidates.csv')
print(f'  Validation saved to: results/top5_validation.csv')
print(f'\nProceed to 06_dashboard.ipynb')